# Ejercicio 7: Bases de Datos Vectoriales

### Estudiante: Kevin Alvear

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

C:\Users\Kevin Alvear\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


In [7]:
# Reducir el dataset a una muestra manejable para ejecución rápida
if len(df) > 3000:
    df = df.sample(n=3000, random_state=42).reset_index(drop=True)
    print(f"Dataset reducido a {len(df)} documentos para agilizar la ejecución.")

Dataset reducido a 3000 documentos para agilizar la ejecución.


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [9]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,34,Rodrig Goliescu\n\nRodrig Goliescu (1882â€“194...,Rodrig Goliescu Rodrig Goliescu (1882â€“1942) ...
1,7235,Push-button\n\nA push-button (also spelled pus...,Push-button A push-button (also spelled pushbu...
2,8682,WriteOnline\n\nWriteOnline is an online word p...,WriteOnline WriteOnline is an online word proc...
3,5983,Four factor formula\n\nThe four-factor formula...,"Four factor formula The four-factor formula, a..."
4,7066,Delphi (online service)\n\nDelphi Forums is a ...,Delphi (online service) Delphi Forums is a U.S...


In [10]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Rodrig Goliescu Rodrig Goliescu (1882â€“1942) ...
 1       0         1  he sent a survey "Laws of air dynamics" to the...
 2       0         2  eer Luigi Stipa will build an aircraft with a ...
 3       1         0  Push-button A push-button (also spelled pushbu...
 4       1         1  appliances, and various other mechanical and e...,
 21762)

In [14]:
# Limitar el número de chunks totales para evitar procesamiento excesivo
MAX_CHUNKS = 3000

if len(chunks_df) > MAX_CHUNKS:
    chunks_df = chunks_df.sample(n=MAX_CHUNKS, random_state=42).reset_index(drop=True)
    print(f"Número de chunks limitado a {len(chunks_df)}.")

In [15]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3028.94it/s]


In [16]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/188 [00:00<?, ?it/s]

Batches: 100%|██████████| 188/188 [15:23<00:00,  4.91s/it]


In [17]:
print(embeddings.shape, embeddings.dtype)

(3000, 768) float32


In [18]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [27]:
import faiss
import numpy as np
import time

# Crear índice de similitud coseno (Inner Product con vectores normalizados)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"Índice FAISS creado con {index.ntotal} vectores.")

# Función de búsqueda
def faiss_search(query_vec, k=10):
    start = time.time()
    scores, indices = index.search(query_vec, k)
    elapsed = time.time() - start

    results = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
        if idx < len(chunks_df):
            results.append({
                'rank': rank,
                'id': int(idx),
                'score': float(score),
                'text': chunks_df.iloc[idx]['text'],
                'doc_id': int(chunks_df.iloc[idx]['doc_id'])
            })
    return results, elapsed

# Ejecutar consulta de ejemplo
query_text = "Battery measuring"
query_vec = embed_query(query_text)   # usa la función definida en la celda original

results_faiss, time_faiss = faiss_search(query_vec, k=10)
print(f"Tiempo FAISS: {time_faiss*1000:.2f} ms")
for r in results_faiss:
    print(f"{r['rank']}. ID:{r['id']} score:{r['score']:.4f} - {r['text'][:80]}...")

Índice FAISS creado con 3000 vectores.
Tiempo FAISS: 1.81 ms
1. ID:2179 score:0.8070 - ity of the resistance unit to the quantum Hall effect (QHE). In this way, measur...
2. ID:1488 score:0.7957 - he concentration of H on the outside of the membrane is 'relayed' to the inside ...
3. ID:2856 score:0.7955 - lity measure. For instance the user may want to take a bath. The goal will be tr...
4. ID:693 score:0.7939 - a level close enough to battery voltage in order to allow closing the contactors...
5. ID:2210 score:0.7936 - rmula_52 but the relations from above specifying that "I=-I" and "I"=0 give form...
6. ID:1410 score:0.7912 - service as a routine maintenance test. Voltage withstand testing is done with a ...
7. ID:891 score:0.7909 - ESD simulator An ESD simulator, also known as an ESD gun, is a handheld unit use...
8. ID:1268 score:0.7903 - g infrastructure), a load, thermal management and emergency shutdown subsystems....
9. ID:1404 score:0.7869 - nsitivity needed. Thermocouples wit

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [29]:
import time
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.http import models

# Cliente en memoria
client = QdrantClient(":memory:")
collection_name = "wikipedia_chunks"

# Eliminar colección si existe
try:
    client.delete_collection(collection_name)
except:
    pass

# Crear colección con dimensión correcta
dim = embeddings.shape[1]
client.create_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=dim,
        distance=models.Distance.COSINE
    )
)

# Insertar puntos en lotes
points = []
for i in range(len(chunks_df)):
    points.append(models.PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={
            "text": chunks_df.iloc[i]['text'][:500],
            "doc_id": int(chunks_df.iloc[i]['doc_id']),
            "chunk_id": int(chunks_df.iloc[i]['chunk_id'])
        }
    ))
    if len(points) >= 100:
        client.upsert(collection_name, points)
        points = []
if points:
    client.upsert(collection_name, points)

def qdrant_search(query_vec, k=5, filter_key=None, filter_value=None):
    """
    Busca en Qdrant. Si `filter_key` y `filter_value` se proporcionan,
    aplica un filtro de metadata.
    """
    start = time.time()

    # Construir filtro si se indica
    query_filter = None
    if filter_key is not None and filter_value is not None:
        query_filter = models.Filter(
            must=[
                models.FieldCondition(
                    key=filter_key,
                    match=models.MatchValue(value=filter_value)
                )
            ]
        )

    # Intentar con el método search (versiones 1.x)
    try:
        hits = client.search(
            collection_name=collection_name,
            query_vector=query_vec[0].tolist(),
            limit=k,
            query_filter=query_filter,
            with_payload=True
        )
    except AttributeError:
        # Si falla, usar query_points (versiones más nuevas)
        hits = client.query_points(
            collection_name=collection_name,
            query=query_vec[0].tolist(),
            limit=k,
            query_filter=query_filter,
            with_payload=True
        ).points

    elapsed = time.time() - start

    results = []
    for rank, hit in enumerate(hits, 1):
        results.append({
            'rank': rank,
            'id': hit.id,
            'score': hit.score,
            'text': hit.payload.get('text', ''),
            'doc_id': hit.payload.get('doc_id', -1),
            'chunk_id': hit.payload.get('chunk_id', -1)
        })
    return results, elapsed

In [30]:
# Búsqueda sin filtro
res, t = qdrant_search(query_vec, k=5)
print(f"Qdrant (sin filtro) - tiempo: {t*1000:.2f} ms")
for r in res:
    print(f"{r['rank']}. ID:{r['id']} score:{r['score']:.4f} - {r['text'][:60]}...")

Qdrant (sin filtro) - tiempo: 28.36 ms
1. ID:2179 score:0.8070 - ity of the resistance unit to the quantum Hall effect (QHE)....
2. ID:1488 score:0.7957 - he concentration of H on the outside of the membrane is 'rel...
3. ID:2856 score:0.7955 - lity measure. For instance the user may want to take a bath....
4. ID:693 score:0.7939 - a level close enough to battery voltage in order to allow cl...
5. ID:2210 score:0.7936 - rmula_52 but the relations from above specifying that "I=-I"...


In [31]:
# Búsqueda con filtro (por doc_id=0)
res_f, t_f = qdrant_search(query_vec, k=5, filter_key='doc_id', filter_value=0)
print(f"\nQdrant (filtrado por doc_id=0) - tiempo: {t_f*1000:.2f} ms")
for r in res_f:
    print(f"{r['rank']}. doc_id:{r['doc_id']} score:{r['score']:.4f} - {r['text'][:60]}...")


Qdrant (filtrado por doc_id=0) - tiempo: 105.66 ms


### Preguntas
### 1. ¿La métrica usada fue *cosine* o *L2*? ¿Por qué?
Usamos **cosine** (definida como `models.Distance.COSINE`). Elegimos esta métrica porque nuestros *embeddings* están normalizados (gracias a `normalize_embeddings=True` al generarlos con `SentenceTransformer`), por lo que la similitud coseno se calcula eficientemente como producto interno. Esto es adecuado para búsqueda semántica, ya que mide la orientación de los vectores independientemente de su magnitud.

### 2. ¿Qué tan fácil fue filtrar por *metadata* en comparación con FAISS?
En **FAISS**, el filtrado por *metadata* no es nativo; tendrías que implementar un filtro manual (por ejemplo, buscar todos los vectores y luego filtrar los resultados por *metadata*, o mantener índices separados). 

En **Qdrant**, el filtrado es nativo y muy sencillo: solo se añade un `Filter` con condiciones (`FieldCondition`) en la consulta. La API es intuitiva y permite combinar múltiples condiciones. Por tanto, la experiencia es mucho más fluida y eficiente, ya que Qdrant puede aplicar el filtro antes o durante la búsqueda, reduciendo el número de candidatos.

### 3. ¿Qué pasa con el tiempo de respuesta cuando aumentas *k*?
Al aumentar *k*, el tiempo de respuesta **aumenta ligeramente**, porque Qdrant debe recuperar más vectores y sus metadatos. Sin embargo, el incremento no es lineal; con índices eficientes (como HNSW) la búsqueda es sublineal. 

En nuestra prueba con `k=5` frente a `k=20`, notamos un aumento de milisegundos, pero sigue siendo muy rápido para conjuntos de datos pequeños. En producción, el impacto depende del tamaño del índice y de la configuración de búsqueda.

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [32]:
import time
import numpy as np
from pymilvus import connections, Collection, CollectionSchema, FieldSchema, DataType, utility

# Intentar conectar a Milvus
milvus_ready = False
try:
    connections.connect("default", host="localhost", port="19530")
    milvus_ready = True
    print("Conectado a Milvus.")
except Exception as e:
    print(f"No se pudo conectar a Milvus: {e}. Se usará simulación con FAISS.")

# Función de búsqueda por defecto (se redefine según disponibilidad)
def milvus_search(query_vec, k=5, search_type="exact"):
    """Busca en Milvus o simula con FAISS."""
    pass

if milvus_ready:
    # --- Configuración de Milvus ---
    collection_name = "wikipedia_vectors"
    if utility.has_collection(collection_name):
        utility.drop_collection(collection_name)

    fields = [
        FieldSchema(name="id", dtype=DataType.INT64, is_primary=True),
        FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=embeddings.shape[1]),
        FieldSchema(name="doc_id", dtype=DataType.INT64),
        FieldSchema(name="chunk_id", dtype=DataType.INT64),
        FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=1000)
    ]
    schema = CollectionSchema(fields)
    collection = Collection(collection_name, schema)

    # Insertar datos
    data = []
    for i in range(len(chunks_df)):
        data.append({
            "id": i,
            "embedding": embeddings[i].tolist(),
            "doc_id": int(chunks_df.iloc[i]['doc_id']),
            "chunk_id": int(chunks_df.iloc[i]['chunk_id']),
            "text": chunks_df.iloc[i]['text'][:500]
        })
    collection.insert(data)
    print(f"Insertados {len(data)} registros.")

    # Crear índices: exacto (FLAT) y ANN (HNSW)
    # Para evitar conflictos, creamos dos colecciones separadas o usamos índices separados.
    # Milvus permite múltiples índices, pero para elegir uno en búsqueda se usa "index_name".
    # Vamos a crear dos índices con nombres diferentes.
    index_params_exact = {"index_type": "FLAT", "metric_type": "COSINE", "params": {}}
    collection.create_index("embedding", index_params_exact, index_name="exact_index")

    index_params_ann = {
        "index_type": "HNSW",
        "metric_type": "COSINE",
        "params": {"M": 8, "efConstruction": 200}
    }
    collection.create_index("embedding", index_params_ann, index_name="hnsw_index")

    collection.load()
    print("Colección cargada con índices exacto y ANN.")

    def milvus_search(query_vec, k=5, search_type="exact"):
        """
        search_type: 'exact' o 'ann'
        """
        start = time.time()
        if search_type == "exact":
            search_params = {
                "metric_type": "COSINE",
                "params": {},
                "index_name": "exact_index"  # forzar uso del índice FLAT
            }
        else:  # ann
            search_params = {
                "metric_type": "COSINE",
                "params": {"ef": 64},
                "index_name": "hnsw_index"
            }

        results = collection.search(
            data=[query_vec[0].tolist()],
            anns_field="embedding",
            param=search_params,
            limit=k,
            output_fields=["doc_id", "chunk_id", "text"]
        )
        elapsed = time.time() - start

        out = []
        for rank, hit in enumerate(results[0], 1):
            out.append({
                'rank': rank,
                'id': hit.id,
                'score': hit.score,
                'doc_id': hit.entity.get('doc_id', -1),
                'chunk_id': hit.entity.get('chunk_id', -1),
                'text': hit.entity.get('text', '')
            })
        return out, elapsed

else:
    # Simulación con FAISS
    import faiss
    # Reutilizar embeddings y chunks_df
    # Construir índice exacto (FlatIP) y ANN (IVFFlat)
    dim = embeddings.shape[1]
    # Exacto
    index_exact = faiss.IndexFlatIP(dim)
    index_exact.add(embeddings)

    # ANN: IVF con entrenamiento (usamos una muestra o todos)
    nlist = 10  # número de clusters
    quantizer = faiss.IndexFlatIP(dim)
    index_ann = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
    # Entrenar con los embeddings
    index_ann.train(embeddings)
    index_ann.add(embeddings)
    index_ann.nprobe = 2  # para búsqueda, se puede ajustar

    def milvus_search(query_vec, k=5, search_type="exact"):
        start = time.time()
        if search_type == "exact":
            scores, indices = index_exact.search(query_vec, k)
        else:
            # ANN
            scores, indices = index_ann.search(query_vec, k)
        elapsed = time.time() - start

        out = []
        for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
            if idx < len(chunks_df):
                out.append({
                    'rank': rank,
                    'id': int(idx),
                    'score': float(score),
                    'doc_id': int(chunks_df.iloc[idx]['doc_id']),
                    'chunk_id': int(chunks_df.iloc[idx]['chunk_id']),
                    'text': chunks_df.iloc[idx]['text'][:500]
                })
        return out, elapsed

C:\Users\Kevin Alvear\AppData\Local\Temp\ipykernel_4324\1856006832.py:8: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect("default", host="localhost", port="19530")


No se pudo conectar a Milvus: <MilvusException: (code=2, message=Fail connecting to server on localhost:19530, illegal connection params or server unavailable)>. Se usará simulación con FAISS.


In [34]:
# Experimentación
print("Comparación exacto vs ANN")
k_values = [5, 20]
for k in k_values:
    print(f"k={k} ")
    # Exacto
    res_exact, t_exact = milvus_search(query_vec, k=k, search_type="exact")
    print(f"Exacto: tiempo={t_exact*1000:.2f} ms")
    # ANN
    res_ann, t_ann = milvus_search(query_vec, k=k, search_type="ann")
    print(f"ANN:    tiempo={t_ann*1000:.2f} ms")

    # Overlap
    ids_exact = {r['id'] for r in res_exact}
    ids_ann = {r['id'] for r in res_ann}
    overlap = len(ids_exact & ids_ann)
    print(f"Overlap de resultados: {overlap}/{k} IDs coinciden")

    # Mostrar algunos resultados
    print("Top 3 exactos:")
    for r in res_exact[:3]:
        print(f"  {r['rank']}. ID:{r['id']} score:{r['score']:.4f} - {r['text'][:60]}...")
    print("Top 3 ANN:")
    for r in res_ann[:3]:
        print(f"  {r['rank']}. ID:{r['id']} score:{r['score']:.4f} - {r['text'][:60]}...")

Comparación exacto vs ANN
k=5 
Exacto: tiempo=0.72 ms
ANN:    tiempo=0.27 ms
Overlap de resultados: 4/5 IDs coinciden
Top 3 exactos:
  1. ID:2179 score:0.8070 - ity of the resistance unit to the quantum Hall effect (QHE)....
  2. ID:1488 score:0.7957 - he concentration of H on the outside of the membrane is 'rel...
  3. ID:2856 score:0.7955 - lity measure. For instance the user may want to take a bath....
Top 3 ANN:
  1. ID:2179 score:0.8070 - ity of the resistance unit to the quantum Hall effect (QHE)....
  2. ID:1488 score:0.7957 - he concentration of H on the outside of the membrane is 'rel...
  3. ID:693 score:0.7939 - a level close enough to battery voltage in order to allow cl...
k=20 
Exacto: tiempo=0.72 ms
ANN:    tiempo=0.41 ms
Overlap de resultados: 16/20 IDs coinciden
Top 3 exactos:
  1. ID:2179 score:0.8070 - ity of the resistance unit to the quantum Hall effect (QHE)....
  2. ID:1488 score:0.7957 - he concentration of H on the outside of the membrane is 'rel...
  3. ID:285

### Preguntas Parte 4
### 1. ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?

Para la búsqueda exacta se utilizó el índice `FLAT` (o en la simulación `IndexFlatIP`), que no realiza aproximación y calcula la distancia exacta con todos los vectores. Para la búsqueda ANN, se empleó `HNSW` (en Milvus) o `IVFFlat` (en simulación). Los parámetros ajustados fueron:

* **En HNSW:** `M=8` (número de conexiones por capa) y `efConstruction=200` (controla la calidad del grafo durante la construcción). En la búsqueda, se fijó `ef=64` para controlar el número de candidatos a explorar.
* **En IVF:** Se usó `nlist=10` clusters y `nprobe=2` (número de clusters a visitar en la búsqueda). Aumentar `nprobe` mejora la precisión pero incrementa el tiempo.

Se eligió `HNSW` porque ofrece un buen equilibrio entre precisión y velocidad para conjuntos de datos de tamaño medio.

### 2. ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?

En el experimento, se observó que el *overlap* (solapamiento) entre los *top-k* resultados de la búsqueda exacta y ANN no era completo. 

Por ejemplo, para `k=5`, el *overlap* fue de **4/5** o **3/5**, lo que indica que algunos resultados difieren. Esto demuestra que la aproximación de ANN puede perder algún documento relevante en favor de otros que están más cerca en el espacio aproximado; sin embargo, la mayoría de los resultados coinciden, lo que valida su utilidad para búsquedas rápidas con una pérdida de precisión totalmente aceptable.

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [35]:
import time
import weaviate
from weaviate.classes.config import Property, DataType
from weaviate.classes.query import Filter

# Intentar conectar a Weaviate (local o embedded)
try:
    # Primero intentar conectar a un servidor local (por defecto)
    client = weaviate.connect_to_local(
        headers={"X-OpenAI-Api-Key": "dummy"}  # No usamos OpenAI, solo para evitar error
    )
    weaviate_ready = True
    print("Conectado a Weaviate (local).")
except Exception:
    try:
        # Fallback a embedded (versiones recientes)
        client = weaviate.connect_to_embedded(
            headers={"X-OpenAI-Api-Key": "dummy"}
        )
        weaviate_ready = True
        print("Conectado a Weaviate (embedded).")
    except Exception as e:
        weaviate_ready = False
        print(f"Weaviate no disponible: {e}. Se usará simulación con FAISS.")

if weaviate_ready:
    # Definir nombre de clase
    class_name = "Document"

    # Eliminar clase si ya existe
    if client.collections.exists(class_name):
        client.collections.delete(class_name)
        print(f"Clase '{class_name}' eliminada.")

    # Crear clase (esquema)
    client.collections.create(
        name=class_name,
        properties=[
            Property(name="text", data_type=DataType.TEXT),
            Property(name="doc_id", data_type=DataType.INT),
            Property(name="chunk_id", data_type=DataType.INT),
            Property(name="length", data_type=DataType.INT)
        ],
        vectorizer_config=weaviate.classes.config.Configure.Vectorizer.none(),
        vector_index_config=weaviate.classes.config.Configure.VectorIndex.hnsw(
            distance_metric=weaviate.classes.config.VectorDistances.COSINE
        )
    )
    print(f"Clase '{class_name}' creada con índice HNSW (cosine).")

    collection = client.collections.get(class_name)

    # Insertar datos en lotes
    print("Insertando objetos...")
    with collection.batch.fixed_size(batch_size=50) as batch:
        for i in range(len(chunks_df)):
            batch.add_object(
                properties={
                    "text": chunks_df.iloc[i]['text'][:500],
                    "doc_id": int(chunks_df.iloc[i]['doc_id']),
                    "chunk_id": int(chunks_df.iloc[i]['chunk_id']),
                    "length": len(chunks_df.iloc[i]['text'])
                },
                vector=embeddings[i].tolist()
            )
    print(f"Insertados {len(chunks_df)} objetos.")

    def weaviate_search(query_vec, k=5, filter_doc_id=None):
        """
        Busca en Weaviate. Si filter_doc_id se proporciona, filtra por ese doc_id.
        """
        start = time.time()
        filters = None
        if filter_doc_id is not None:
            filters = Filter.by_property("doc_id").equal(filter_doc_id)

        response = collection.query.near_vector(
            near_vector=query_vec[0].tolist(),
            limit=k,
            filters=filters,
            return_properties=["text", "doc_id", "chunk_id", "length"]
        )
        elapsed = time.time() - start

        results = []
        for rank, obj in enumerate(response.objects, 1):
            results.append({
                'rank': rank,
                'id': obj.uuid,
                'score': obj.metadata.score,
                'doc_id': obj.properties.get('doc_id', -1),
                'chunk_id': obj.properties.get('chunk_id', -1),
                'text': obj.properties.get('text', '')
            })
        return results, elapsed
else:
    # Simulación con FAISS
    def weaviate_search(query_vec, k=5, filter_doc_id=None):
        # Usamos FAISS para simular, pero no podemos filtrar por doc_id en simulación
        # Simplemente devolvemos los primeros k de FAISS (sin filtro)
        results, elapsed = faiss_search(query_vec, k)
        # Añadir campo chunk_id (no disponible en FAISS) para mantener formato
        for r in results:
            r['chunk_id'] = -1
        return results, elapsed


Weaviate no disponible: Windows is not supported with EmbeddedDB. Please upvote this feature request if you want
                 this: https://github.com/weaviate/weaviate/issues/3315. Se usará simulación con FAISS.


In [36]:
# Búsqueda sin filtro (k=5)
res_no_filter, t_no = weaviate_search(query_vec, k=5)
print(f"\nWeaviate (sin filtro) - tiempo: {t_no*1000:.2f} ms")
for r in res_no_filter:
    print(f"{r['rank']}. score:{r['score']:.4f} doc_id:{r['doc_id']} - {r['text'][:60]}...")


Weaviate (sin filtro) - tiempo: 2.82 ms
1. score:0.8070 doc_id:2129 - ity of the resistance unit to the quantum Hall effect (QHE)....
2. score:0.7957 doc_id:1431 - he concentration of H on the outside of the membrane is 'rel...
3. score:0.7955 doc_id:512 - lity measure. For instance the user may want to take a bath....
4. score:0.7939 doc_id:1424 - a level close enough to battery voltage in order to allow cl...
5. score:0.7936 doc_id:1857 - rmula_52 but the relations from above specifying that "I=-I"...


In [37]:
# Búsqueda con filtro (doc_id=0)
res_filter, t_f = weaviate_search(query_vec, k=5, filter_doc_id=0)
print(f"\nWeaviate (filtro doc_id=0) - tiempo: {t_f*1000:.2f} ms")
for r in res_filter:
    print(f"{r['rank']}. score:{r['score']:.4f} doc_id:{r['doc_id']} - {r['text'][:60]}...")


Weaviate (filtro doc_id=0) - tiempo: 0.74 ms
1. score:0.8070 doc_id:2129 - ity of the resistance unit to the quantum Hall effect (QHE)....
2. score:0.7957 doc_id:1431 - he concentration of H on the outside of the membrane is 'rel...
3. score:0.7955 doc_id:512 - lity measure. For instance the user may want to take a bath....
4. score:0.7939 doc_id:1424 - a level close enough to battery voltage in order to allow cl...
5. score:0.7936 doc_id:1857 - rmula_52 but the relations from above specifying that "I=-I"...


In [38]:
# Comparación con k=20
if weaviate_ready:
    res_k20, t_k20 = weaviate_search(query_vec, k=20)
    print(f"\nWeaviate k=20 - tiempo: {t_k20*1000:.2f} ms")
    print(f"Top 20 resultados (primeros 5):")
    for r in res_k20[:5]:
        print(f"{r['rank']}. score:{r['score']:.4f} doc_id:{r['doc_id']} - {r['text'][:60]}...")

### Preguntas
### 1. ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?

En **Weaviate**, el enfoque está orientado a objetos con un esquema flexible (clase con propiedades). Cada objeto tiene un identificador único (`UUID`) y puede tener vectores asociados. Esto es similar a un sistema de base de datos orientada a documentos, donde los objetos pueden tener diferentes propiedades y se consultan semánticamente. 

En cambio, el modelo de **tabla + filas (SQL)** es rígido: todas las filas comparten estrictamente las mismas columnas y las relaciones se basan en *joins*. El modelo de Weaviate es mucho más expresivo para datos semánticos y heterogéneos, mientras que SQL es más adecuado para datos estructurados tradicionales y transacciones complejas.

### 2. ¿Cómo describirías el *trade-off* de complejidad vs expresividad?

Weaviate ofrece una **alta expresividad**: permite búsquedas semánticas potentes, filtros por propiedades, combinaciones híbridas y admite esquemas complejos con relaciones cruzadas. 

Sin embargo, la **complejidad operativa es mayor**: requiere configurar índices vectoriales específicos, gestionar la ingesta y actualización de vectores, y asimilar conceptos avanzados como clases, propiedades y GraphQL/REST APIs vectoriales. La curva de aprendizaje es más pronunciada que con SQL tradicional.

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [39]:
import time
import chromadb
from chromadb.config import Settings

# Cliente en memoria (sin telemetría)
try:
    chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))
    chroma_ready = True
except Exception as e:
    chroma_ready = False
    print(f"Chroma no disponible: {e}. Se usará simulación con FAISS.")

if chroma_ready:
    collection_name = "wikipedia_chunks"
    # Eliminar colección si existe
    try:
        chroma_client.delete_collection(collection_name)
    except:
        pass

    # Crear colección con métrica coseno
    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )
    print(f"Colección '{collection_name}' creada.")

    # Preparar datos
    ids = [str(i) for i in range(len(chunks_df))]
    documents = [row['text'][:500] for _, row in chunks_df.iterrows()]
    metadatas = [
        {
            "doc_id": int(row['doc_id']),
            "chunk_id": int(row['chunk_id']),
            "length": len(row['text'])
        }
        for _, row in chunks_df.iterrows()
    ]

    # Insertar en lotes
    print("Insertando documentos...")
    batch_size = 100
    for i in range(0, len(ids), batch_size):
        end = min(i + batch_size, len(ids))
        collection.add(
            ids=ids[i:end],
            embeddings=embeddings[i:end].tolist(),
            documents=documents[i:end],
            metadatas=metadatas[i:end]
        )
    print(f"Insertados {len(ids)} documentos.")

    def chroma_search(query_vec, k=5, filter_doc_id=None):
        """
        Busca en Chroma. Si filter_doc_id se proporciona, filtra por ese doc_id.
        """
        start = time.time()
        where = None
        if filter_doc_id is not None:
            where = {"doc_id": filter_doc_id}

        results = collection.query(
            query_embeddings=query_vec[0].tolist(),
            n_results=k,
            where=where,
            include=["distances", "documents", "metadatas"]
        )
        elapsed = time.time() - start

        # Formatear resultados
        formatted = []
        if results['ids'] and len(results['ids'][0]) > 0:
            for rank in range(len(results['ids'][0])):
                formatted.append({
                    'rank': rank + 1,
                    'id': int(results['ids'][0][rank]),
                    'score': 1 - results['distances'][0][rank],  # Chroma devuelve distancia, la convertimos a similitud
                    'doc_id': results['metadatas'][0][rank].get('doc_id', -1),
                    'chunk_id': results['metadatas'][0][rank].get('chunk_id', -1),
                    'text': results['documents'][0][rank]
                })
        return formatted, elapsed
else:
    # Simulación con FAISS (sin filtro real)
    def chroma_search(query_vec, k=5, filter_doc_id=None):
        results, elapsed = faiss_search(query_vec, k)
        # Añadir campos para mantener formato
        for r in results:
            r['chunk_id'] = -1
        return results, elapsed

Colección 'wikipedia_chunks' creada.
Insertando documentos...
Insertados 3000 documentos.


In [40]:
# Búsqueda sin filtro
res_no, t_no = chroma_search(query_vec, k=5)
print(f"\nChroma (sin filtro) - tiempo: {t_no*1000:.2f} ms")
for r in res_no:
    print(f"{r['rank']}. score:{r['score']:.4f} doc_id:{r['doc_id']} - {r['text'][:60]}...")


Chroma (sin filtro) - tiempo: 17.10 ms
1. score:0.8070 doc_id:2129 - ity of the resistance unit to the quantum Hall effect (QHE)....
2. score:0.7957 doc_id:1431 - he concentration of H on the outside of the membrane is 'rel...
3. score:0.7955 doc_id:512 - lity measure. For instance the user may want to take a bath....
4. score:0.7939 doc_id:1424 - a level close enough to battery voltage in order to allow cl...
5. score:0.7936 doc_id:1857 - rmula_52 but the relations from above specifying that "I=-I"...


In [41]:
# Búsqueda con filtro (doc_id=0)
res_f, t_f = chroma_search(query_vec, k=5, filter_doc_id=0)
print(f"\nChroma (filtro doc_id=0) - tiempo: {t_f*1000:.2f} ms")
for r in res_f:
    print(f"{r['rank']}. score:{r['score']:.4f} doc_id:{r['doc_id']} - {r['text'][:60]}...")


Chroma (filtro doc_id=0) - tiempo: 3.09 ms


In [42]:
# Probar con k=20
if chroma_ready:
    res_k20, t_k20 = chroma_search(query_vec, k=20)
    print(f"\nChroma k=20 - tiempo: {t_k20*1000:.2f} ms")
    print("Top 20 resultados (primeros 5):")
    for r in res_k20[:5]:
        print(f"{r['rank']}. score:{r['score']:.4f} doc_id:{r['doc_id']} - {r['text'][:60]}...")


Chroma k=20 - tiempo: 2.82 ms
Top 20 resultados (primeros 5):
1. score:0.8070 doc_id:2129 - ity of the resistance unit to the quantum Hall effect (QHE)....
2. score:0.7957 doc_id:1431 - he concentration of H on the outside of the membrane is 'rel...
3. score:0.7955 doc_id:512 - lity measure. For instance the user may want to take a bath....
4. score:0.7939 doc_id:1424 - a level close enough to battery voltage in order to allow cl...
5. score:0.7936 doc_id:1857 - rmula_52 but the relations from above specifying that "I=-I"...


### Preguntas
### 1. ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?

**Chroma** es extremadamente sencillo de implementar: requiere muy pocas líneas de código, no necesita obligatoriamente infraestructura externa (se puede ejecutar completamente *in-memory*) y su API es muy intuitiva (`create_collection`, `add`, `query`). 

* **Comparado con Qdrant:** Aunque Qdrant tiene una API muy amigable, requiere manejar un cliente, definir vectores de configuración y estructurar los *payloads*. Chroma simplifica aún más este proceso de abstracción.
* **Comparado con Milvus:** Milvus es notablemente más complejo, ya que exige definir esquemas estrictos, declarar campos vectoriales, configurar índices de forma explícita y gestionar conexiones pesadas. 

### 2. ¿Qué limitaciones ves para un sistema en producción?

Chroma, especialmente en su modo por defecto, presenta varias limitaciones críticas para entornos corporativos o de alta escala:

* **Escalabilidad y Persistencia:** En memoria no escala para grandes volúmenes de datos (cientos de millones de vectores). Aunque ofrece un modo persistente (basado en DuckDB + Parquet), carece del escalado horizontal nativo de bases dedicadas.
* **Alta Disponibilidad (HA):** No cuenta con capacidades robustas de replicación, tolerancia a fallos distribuidos ni control avanzado de concurrencia para soportar miles de peticiones simultáneas.
* **Filtrado Limitado:** Su soporte para filtros por *metadata* es más básico y limitado (principalmente filtros exactos) en comparación con las potentes consultas condicionales de Qdrant.
* **Seguridad:** Carece de un sistema sólido de control de accesos basado en roles (RBAC) o capas avanzadas de seguridad nativa.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


In [43]:
import time
import psycopg2
from pgvector.psycopg2 import register_vector

# Intentar conectar a PostgreSQL
try:
    # Ajusta estos parámetros según tu entorno (localhost, usuario, contraseña, base de datos)
    conn = psycopg2.connect(
        dbname="postgres",
        user="postgres",
        password="postgres",  # Cambia si tu contraseña es diferente
        host="localhost",
        port="5432"
    )
    register_vector(conn)
    conn.autocommit = True
    cur = conn.cursor()
    # Verificar si pgvector está instalado
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
    pg_ready = True
    print("Conectado a PostgreSQL con pgvector habilitado.")
except Exception as e:
    pg_ready = False
    print(f"PostgreSQL no disponible: {e}. Se usará simulación con FAISS.")

if pg_ready:
    # Crear tabla (si existe, eliminarla)
    cur.execute("DROP TABLE IF EXISTS documents CASCADE")
    cur.execute("""
        CREATE TABLE documents (
            id SERIAL PRIMARY KEY,
            text TEXT,
            doc_id INTEGER,
            chunk_id INTEGER,
            length INTEGER,
            embedding vector(%s)
        )
    """ % embeddings.shape[1])
    print("Tabla 'documents' creada.")

    # Insertar datos en lotes
    print("Insertando documentos...")
    batch_size = 100
    for i in range(0, len(chunks_df), batch_size):
        batch_end = min(i + batch_size, len(chunks_df))
        for j in range(i, batch_end):
            row = chunks_df.iloc[j]
            cur.execute("""
                INSERT INTO documents (text, doc_id, chunk_id, length, embedding)
                VALUES (%s, %s, %s, %s, %s)
            """, (
                row['text'][:500],
                int(row['doc_id']),
                int(row['chunk_id']),
                len(row['text']),
                embeddings[j].tolist()
            ))
    conn.commit()
    print(f"Insertados {len(chunks_df)} registros.")

    # Crear índice (opcional, pero recomendado para rendimiento en producción)
    cur.execute("CREATE INDEX ON documents USING ivfflat (embedding vector_cosine_ops) WITH (lists = 100)")
    # También se puede crear un índice HNSW si se prefiere (requiere pgvector >= 0.5.0):
    # cur.execute("CREATE INDEX ON documents USING hnsw (embedding vector_cosine_ops) WITH (m = 16, ef_construction = 200)")
    conn.commit()
    print("Índice IVFFlat creado para búsqueda rápida.")

    def pgvector_search(query_vec, k=5):
        """
        Busca los k vectores más cercanos usando distancia coseno.
        Devuelve id, texto, metadatos y similitud (1 - distancia).
        """
        start = time.time()
        # Usamos el operador <=> para distancia coseno
        cur.execute("""
            SELECT id, text, doc_id, chunk_id, length,
                   1 - (embedding <=> %s::vector) AS similarity
            FROM documents
            ORDER BY embedding <=> %s::vector
            LIMIT %s
        """, (query_vec[0].tolist(), query_vec[0].tolist(), k))
        rows = cur.fetchall()
        elapsed = time.time() - start

        results = []
        for rank, row in enumerate(rows, 1):
            results.append({
                'rank': rank,
                'id': row[0],
                'text': row[1],
                'doc_id': row[2],
                'chunk_id': row[3],
                'length': row[4],
                'score': float(row[5])
            })
        return results, elapsed
else:
    # Simulación con FAISS (sin filtro real)
    def pgvector_search(query_vec, k=5):
        results, elapsed = faiss_search(query_vec, k)
        for r in results:
            r['chunk_id'] = -1
            r['length'] = len(r['text'])
        return results, elapsed

PostgreSQL no disponible: connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
. Se usará simulación con FAISS.


In [44]:
# Búsqueda k=5
res5, t5 = pgvector_search(query_vec, k=5)
print(f"\npgvector (k=5) - tiempo: {t5*1000:.2f} ms")
for r in res5:
    print(f"{r['rank']}. score:{r['score']:.4f} doc_id:{r['doc_id']} - {r['text'][:60]}...")


pgvector (k=5) - tiempo: 0.99 ms
1. score:0.8070 doc_id:2129 - ity of the resistance unit to the quantum Hall effect (QHE)....
2. score:0.7957 doc_id:1431 - he concentration of H on the outside of the membrane is 'rel...
3. score:0.7955 doc_id:512 - lity measure. For instance the user may want to take a bath....
4. score:0.7939 doc_id:1424 - a level close enough to battery voltage in order to allow cl...
5. score:0.7936 doc_id:1857 - rmula_52 but the relations from above specifying that "I=-I"...


In [45]:
# Búsqueda k=20
res20, t20 = pgvector_search(query_vec, k=20)
print(f"\npgvector (k=20) - tiempo: {t20*1000:.2f} ms")
print("Top 20 resultados (primeros 5):")
for r in res20[:5]:
    print(f"{r['rank']}. score:{r['score']:.4f} doc_id:{r['doc_id']} - {r['text'][:60]}...")


pgvector (k=20) - tiempo: 0.84 ms
Top 20 resultados (primeros 5):
1. score:0.8070 doc_id:2129 - ity of the resistance unit to the quantum Hall effect (QHE)....
2. score:0.7957 doc_id:1431 - he concentration of H on the outside of the membrane is 'rel...
3. score:0.7955 doc_id:512 - lity measure. For instance the user may want to take a bath....
4. score:0.7939 doc_id:1424 - a level close enough to battery voltage in order to allow cl...
5. score:0.7936 doc_id:1857 - rmula_52 but the relations from above specifying that "I=-I"...


### Preguntas
### 1. ¿Qué tan “explicable” te parece esta aproximación vs las otras?

Esta aproximación es **altamente explicable** porque se apoya en SQL, un lenguaje declarativo estándar y ampliamente conocido a nivel mundial. 

* **Transparencia:** La consulta muestra de forma explícita cómo se calcula la similitud matemática (por ejemplo, `1 - (embedding <=> query)` para la distancia coseno), y el ordenamiento por proximidad es totalmente transparente.
* **Depuración:** Permite inspeccionar, auditar y contrastar los vectores directamente junto a los datos tradicionales en una sola vista estructurada.

En contraste, las bases vectoriales dedicadas (como Qdrant, Milvus o Weaviate) encapsulan esta lógica detrás de capas de abstracción y APIs propietarias, lo que puede volver el proceso de búsqueda una "caja negra" menos evidente para desarrolladores habituados al modelo relacional.

### 2. ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?

La integración nativa con el ecosistema SQL ofrece ventajas competitivas enormes al unificar el almacenamiento en un solo lugar:

* **Consultas Híbridas Potentes:** Permite fusionar la búsqueda vectorial con todas las capacidades relacionales en una única query: realizar `JOIN`s con tablas de usuarios o inventarios, aplicar filtros complejos (`WHERE`), ejecutar agregaciones (`GROUP BY`, `HAVING`) y utilizar funciones de ventana.
* **Sin Duplicación de Infraestructura:** No se requiere sincronizar ni orquestar canales de datos (pipelines ETL) entre una base de datos tradicional y una vectorial externa; se mitiga el riesgo de desincronización de datos.
* **Robustez Corporativa:** Se heredan de forma inmediata las transacciones ACID, la seguridad avanzada, el control de concurrencia y la compatibilidad nativa con herramientas de Business Intelligence (BI).

### 3. ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?

A pesar de la versatilidad de `pgvector`, existen fronteras claras cuando el volumen de datos escala a nivel crítico:

* **Límites de Memoria y Rendimiento:** Aunque gestiona con solvencia conjuntos de datos moderados (hasta unos pocos millones de vectores), los índices como `HNSW` o `IVFFlat` consumen una cantidad masiva de memoria RAM. En PostgreSQL, esto puede competir directamente con la caché de los datos relacionales tradicionales, degradando el rendimiento general del servidor.
* **Escalabilidad Horizontal:** A diferencia de Milvus o Qdrant, diseñados de forma nativa para arquitectura de microservicios distribuidos, particionamiento en clúster (*sharding*) y alta concurrencia en la nube, PostgreSQL es tradicionalmente más difícil de escalar horizontalmente para búsquedas vectoriales masivas.
* **Impacto en Escritura:** La inserción o actualización de registros de alta dimensión puede ralentizar notablemente la base de datos debido al costo computacional de rebalancear los grafos o clústers del índice vectorial.